In [1]:
import pandas as pd
matches = pd.read_csv('data/atp_matches_2023.csv')
print(matches.shape)
matches.head()

(2986, 49)


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2023-9900,United Cup,Hard,18,A,20230102,300,126203,3.0,NaN,...,62.0,47.0,15.0,12.0,9.0,9.0,9.0,3355.0,16.0,2375.0
1,2023-9900,United Cup,Hard,18,A,20230102,299,126207,NaN,NaN,...,12.0,8.0,3.0,4.0,1.0,3.0,19.0,2000.0,23.0,1865.0
2,2023-9900,United Cup,Hard,18,A,20230102,296,126203,3.0,NaN,...,62.0,51.0,7.0,12.0,2.0,2.0,9.0,3355.0,10.0,2905.0
3,2023-9900,United Cup,Hard,18,A,20230102,295,126207,NaN,NaN,...,41.0,26.0,12.0,9.0,6.0,9.0,19.0,2000.0,245.0,220.0
4,2023-9900,United Cup,Hard,18,A,20230102,292,126774,1.0,NaN,...,58.0,48.0,18.0,16.0,1.0,2.0,4.0,5550.0,16.0,2375.0


In [2]:
# all 2021 tour matches
matches = pd.read_csv('data/atp_matches_2021.csv')
# all the 2021 challenger/qualifying matches
chall = pd.read_csv('data/atp_matches_qual_chall_2021.csv')
# stacking both fo those (matches and chall) into one table
all_matches_2021 = pd.concat([matches, chall], ignore_index=True)


print(all_matches_2021.shape)

(10230, 49)


In [3]:
# takes just the winner and loser ID columns from all 2021 matches, and make a separate copy
winners_view = all_matches_2021[['winner_id', 'loser_id']].copy()
# relabel the columns from loser and winner to opponent and player
winners_view = winners_view.rename(columns={'winner_id': 'player_id', 'loser_id': 'opponent_id'})
# mark Won = true  for the person who won
winners_view['won'] = True

winners_view.head()

,player_id,opponent_id,won
0,126207,126952,True
1,105526,106329,True
2,111576,104797,True
3,105357,207518,True
4,207830,105311,True


In [4]:
# take just the winner and loser Id columns again
losers_view = all_matches_2021[['winner_id', 'loser_id']].copy()
# relabel columns (won describes player_id)
losers_view = losers_view.rename(columns={'loser_id': 'player_id', 'winner_id': 'opponent_id'})
losers_view['won'] = False
losers_view.head()

,opponent_id,player_id,won
0,126207,126952,False
1,105526,106329,False
2,111576,104797,False
3,105357,207518,False
4,207830,105311,False


In [27]:
# stack winner and losers table into one
player_matches = pd.concat([winners_view, losers_view], ignore_index=True)
print(player_matches.shape)

(20460, 4)


In [29]:
player_matches.head()

,player_id,opponent_id,tourney_date,won
0,126207,126952,20210724,True
1,105526,106329,20210724,True
2,111576,104797,20210724,True
3,105357,207518,20210724,True
4,207830,105311,20210724,True


In [30]:
players = pd.read_csv('data/atp_players.csv')
players[players['name_last'] == 'Alcaraz']  # filter by last name to find alcaraz

/var/folders/rk/w0plzk6n4033x_g087swl6t40000gn/T/ipykernel_63018/3246681681.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  players = pd.read_csv('data/atp_players.csv')


,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
3577,103578,Antonio,Alcaraz,U,19800621.0,ESP,NaN,NaN
23371,123387,Arius,Alcaraz,U,19720825.0,PHI,NaN,NaN
60275,207989,Carlos,Alcaraz,R,20030505.0,ESP,183.0,Q85518537


In [15]:
# group all rows by player_id, then average the won column (True=1, False=0)
win_rates = player_matches.groupby('player_id')['won'].mean()

# look up a players win rate
print("Carlos Alcaraz win rate is: ", win_rates.loc[207989])

Carlos Alcaraz win rate is:  0.7164179104477612


In [34]:
# add the tournament date for each player in player_matches
player_matches['tourney_date'] = pd.to_datetime(player_matches['tourney_date'], format='%Y%m%d')

print(player_matches.shape)
player_matches.head()

(20460, 4)


,player_id,opponent_id,tourney_date,won
0,126207,126952,2021-07-24,True
1,105526,106329,2021-07-24,True
2,111576,104797,2021-07-24,True
3,105357,207518,2021-07-24,True
4,207830,105311,2021-07-24,True


In [38]:
import glob

# find every ranking file in the data folder
rank_files = glob.glob('data/atp_rankings_*.csv')

# load all of the files and stack them into one big rankings table
rankings = pd.concat([pd.read_csv(f) for f in rank_files], ignore_index=True)

# convert the ranking date into a readable format
rankings['ranking_date'] = pd.to_datetime(rankings['ranking_date'], format='%Y%m%d')

# sort each player match by the tournament date
player_matches_sorted = player_matches.sort_values('tourney_date')

# sort the rankings of each player by the ranking date
rankings_sorted = rankings.sort_values('ranking_date')

player_matches_sorted.head()


,player_id,opponent_id,tourney_date,won
3604,105933,105271,2021-01-04,True
1160,126205,106227,2021-01-04,True
1161,111815,111574,2021-01-04,True
1162,106216,110536,2021-01-04,True
1163,126207,105385,2021-01-04,True


In [39]:
player_matches_sorted.dtypes

player_id                int64
opponent_id              int64
tourney_date    datetime64[ns]
won                       bool
dtype: object

In [40]:
# merge each match with the opponent's most recent ranking as of that match date
# look for every match what the opponent's ranking was on the most recent date before the match
# find the opponents rank without needing an exact match date
opponent_ranks = pd.merge_asof(
    player_matches_sorted, # the 1st table (players) (left table)
    rankings_sorted, # the 2nd table (lookup table for rankings) (right table)
    left_on = 'tourney_date', # in table 1, find the date
    right_on = 'ranking_date', # in table 2, find the date
    left_by = 'opponent_id', # lookup opponent in table 1
    right_by = 'player', # look for player that opponent_id matches
    direction = 'backward' # for ex, if march 25 tourney happened, choose the march 22nd date instead of march 29th
)

opponent_ranks.head()
    

,player_id,opponent_id,tourney_date,won,ranking_date,rank,player,points
0,105933,105271,2021-01-04,True,2021-01-04,435.0,105271.0,87.0
1,126205,106227,2021-01-04,True,2021-01-04,268.0,106227.0,205.0
2,111815,111574,2021-01-04,True,2021-01-04,299.0,111574.0,165.0
3,106216,110536,2021-01-04,True,2021-01-04,304.0,110536.0,154.0
4,126207,105385,2021-01-04,True,2021-01-04,327.0,105385.0,135.0


In [41]:
opponent_ranks[opponent_ranks['opponent_id'] == 207989][['opponent_id', 'tourney_date', 'rank']].head()

,opponent_id,tourney_date,rank
176,207989,2021-01-10,141.0
181,207989,2021-01-10,141.0
247,207989,2021-01-10,141.0
891,207989,2021-02-01,146.0
914,207989,2021-02-01,146.0
